# Spatial signature analysis

This notebook showcases the spatial signature analysis feature introduced in pylandstats v3.1.0. As an example use case, we explore an inventory of landscapes across the canton of Vaud (Switzerland), with the aim of characterizing them by means of their spatial signatures - namely a numerical embedding describing a landscape pattern. Examples of spatial signatures include a vector with the proportion of landscape occupied by each class, co-occurrence matrices reflecting pairwise adjacencies between classes or to vector of computed landscape metrics - see the [motif R package](https://github.com/Nowosad/motif) and its journal article [1] for more details on spatial signatures.

Given a set of landscapes, their spatial signatures can be used to perform operations such as spatial pattern search, change detection or clustering (see Nowosad [1] for more details). In this example, spatial signatures are used to cluster landscapes and obtain a typology of Vaud landscapes. The cells of the data processing section serve to generate an inventory of local landscapes across the canton of Vaud. The landscape inventory is used to showcase how the `SpatialSignatureAnalysis` class can be used to explore the fundamental components of landscape metrics as well as to cluster landscapes. Finally, the landscape inventory is clustered using two types of spatial signatures, namely (a) a vector of ten recurrent landscape metrics and (b) an information theory (IT) approach based on Nowosad and Stepinski [2].

We will begin with some imports, definitions and data processing. If you are mainly interested in the features of `SpatialSignatureAnalysis`, feel free to skip to ["2. Spatial signature analysis"](#spatial-signature-analysis) section.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio as rio
import seaborn as sns
import swisslandstats as sls
from rasterio import features
from shapely import geometry
from sklearn import decomposition

import pylandstats as pls

# set parameters which are not related to the cluster analysis itself (e.g., plotting)
# random seed to ensure repeatability of this notebook
random_seed = 0
# TODO: use random.PCG64 - see https://github.com/scikit-learn/scikit-learn/issues/16988
# bg = random.RandomState(random_seed)

# plotting parameters
figwidth, figheight = plt.rcParams["figure.figsize"]

heatmap_kwargs = dict(annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)

# parameters for the cluster landscape plots
plot_cluster_landscapes_kwargs = dict(
    figsize=(3, 3),
    sample_kwargs=dict(random_state=random_seed),
    subfigures_kwargs=dict(hspace=0.05),
    supylabel_kwargs=dict(x=0.1),
    cmap=sls.noas04_4_cmap,
    norm=sls.noas04_4_norm,
)


def plot_cgram_eval(cgram, *, eval_methods=None):
    """Plot clustergram evaluation methods."""
    if eval_methods is None:
        eval_methods = [
            "silhouette_score",
            "calinski_harabasz_score",
            "davies_bouldin_score",
        ]
    n_plots = len(eval_methods)
    fig, axes = plt.subplots(n_plots, 1, figsize=(figwidth, figheight * n_plots))
    for eval_method, ax in zip(eval_methods, axes):
        getattr(cgram, eval_method)().plot(ax=ax)
        ax.set_ylabel(eval_method)

    ax.set_xlabel("n. clusters")

    return fig

## 1. Data preprocessing

The land use/land cover (LULC) data used in this notebook is a clip of the Swiss Land Statistics (SLS) survey over the canton of Vaud, which is shipped with the docs in the `data/vaud` directory. See the [pylandstats-notebooks](https://github.com/martibosch/pylandstats-notebooks) repository for the preprocessing pipeline that derives it from the raw SLS data.

We will set the main parameters of the cluster analysis in the cell below:

In [ ]:
lulc_col = "LU18_4"
input_filepath = f"data/vaud/{lulc_col}.tif"

# Size (in meters) of each landscape, i.e., each landscape is a tile of 4000x4000 m^2,
# i.e., 4x4 km^2
landscape_size = 4000

Let us start by generating the landscapes for the spatial signature analysis by generating a `ZonalGridAnalysis` with the target grid size. However, in order to enhance landscape comparability, we will exclude the grid cells at the border of our raster extent so that we only have squared landscapes full of valid data pixels:

In [ ]:
zga = pls.ZonalGridAnalysis(
    input_filepath,
    zone_width=landscape_size,
    zone_height=landscape_size,
    offset="center",
)
with rio.open(input_filepath) as src:
    extent_geom = gpd.GeoSeries(
        [
            geometry.shape(geom)
            for geom, val in features.shapes(
                src.dataset_mask(), transform=src.transform
            )
            if val != src.nodata
        ],
        crs=src.crs,
    ).union_all()

is_inner = zga.zone_gser.within(extent_geom)
ax = gpd.GeoDataFrame({"is_inner": is_inner}, geometry=zga.zone_gser).plot(
    column="is_inner", edgecolor="black", legend=True
)
for geom in extent_geom.geoms:
    ax.plot(*geom.exterior.xy, color="orange")

Let us now use the filtered geo-series of grid cells to instantiate a `ZonalAnalysis` with only the landscapes that are fully contained by the raster extent (i.e., the cantonal border):

In [ ]:
za = pls.ZonalAnalysis(input_filepath, zga.zone_gser[is_inner].copy())

Here is what one of this landscapes looks like (legend: red pixels are urban, green pixels are agricultural, yellow pixels are wooded areas and blue pixels are unproductive areas, e.g., lakes, rivers, glaciers...):

In [ ]:
za.landscape_ser.sample(1, random_state=random_seed).iloc[0].plot_landscape(
    cmap=sls.noas04_4_cmap, norm=sls.noas04_4_norm, legend=True
)

(spatial-signature-analysis)=
## Spatial signature analysis

We can now use the generated `ZonalAnalysis` instantiate the `SpatialSignatureAnalysis` class. In fact, we could also use any other pylandstats multi-landscape class (e.g., `SpatioTemporalAnalysis`, `SpatioTemporalZonalAnalysis`, `ZonalGridAnalysis`...). However, unlike the other pylandstats multi-landscape classes, in a `SpatialSignatureAnalysis` the metrics are computed when the object is instantiated. Therefore, the initialization requires the list of target metrics both for the class and landscape level:

In [ ]:
ssa = pls.SpatialSignatureAnalysis(
    za,
    class_metrics=[
        "proportion_of_landscape",
        "edge_density",
    ],
    landscape_metrics=[
        "shannon_diversity_index",
    ],
)

The computed metrics for each landscape can be accessed via the `metrics_df` attribute, which is definitive and thus cannot be changed after the instantiation:

In [ ]:
ssa.metrics_df.head()

As shown above, it is possible to include both metrics at the class and landscape level in the spatial signature by providing both the `class_metrics` and `landscape_metrics` arguments. Instead of using a multi-level index with the landscape id and class value (like in the other pylandstats multi-landscape classes), the class values are ["pivoted"](https://pandas.pydata.org/docs/user_guide/reshaping.html) into the columns so that each row is a vector of metrics, i.e., the spatial signature of the landscape.

Therefore, the resulting data frame consists of a single row unique to each landscape, which features all the computed metrics (at the class and landscape-level) as columns, i.e., the spatial signature of the landscape.

Likewise the other pylandstats multi-landscape classes, we can use the `classes` argument compute the metrics for a subset of classes only. Similarly, it is possible to customize how the metrics are computed, however, in `SpatialSignatureAnalysis.compute_metrics_df` this is done by means of two arguments `class_metrics_kwargs` and `landscape_metrics_kwargs`, which customize the computation of the class and landscape-level metrics, respectively.

In [ ]:
ssa = pls.SpatialSignatureAnalysis(
    za,
    class_metrics=[
        "proportion_of_landscape",
    ],
    classes=[1, 2],
    class_metrics_kwargs={"proportion_of_landscape": {"percent": False}},
    landscape_metrics=[
        "edge_density",
        "shannon_diversity_index",
    ],
    landscape_metrics_kwargs={
        "edge_density": {"count_boundary": True},
    },
)
ssa.metrics_df.head()

Further details about the arguments of the `SpatialSignatureAnalysis` initialization can be found in the [API documentation](https://pylandstats.readthedocs.io/en/latest/spatial-signatures.html).

Let us now focus on the (many) potential applications of the spatial signature analysis. From a data science perspective, the `metrics_df` constitutes a dataset matrix in which each row is a sample (i.e., a landscape) that is in turn represented by a feature vector (i.e., a metric). This dataset matrix can be used for a wide range of computational landscape ecology applications, such as clustering similar landscapes or identifying the main components of spatial patterns [3]. In Python, the [scikit-learn library](https://scikit-learn.org/stable/index.html) [4] provides a wide range of tools for data science and machine learning that can be used for these purposes.

The sections below show how the `SpatialSignatureAnalysis` provides a convenient interface to use scikit-learn tools for clustering and component analysis of spatial patterns. Let us start by instantiating a `SpatialSignatureAnalysis` with the generated landscapes and a set of ten metrics (at the landscape level only) chosen loosely following the work of Nowosad and Stepinski [5] (the list of metrics is actually adapted considering the metrics that are currently implemented in pylandstats):

In [ ]:
ten_metrics = [
    # area and edge
    "area_mn",
    "perimeter_mn",
    "patch_density",
    "edge_density",
    # shape
    "fractal_dimension_am",
    "shape_index_mn",
    # aggregation
    "contagion",
    "effective_mesh_size",
    "landscape_shape_index",
    # diversity
    "shannon_diversity_index",
]
ten_ssa = pls.SpatialSignatureAnalysis(
    za,
    landscape_metrics=ten_metrics,
)
ten_ssa.metrics_df.head()

## Component analysis

As extensively reviewed in the literature, landscape metrics are highly correlated, which can be problematic for many applications, e.g., multicolinearity can undermine statistical inference when establishing  relationships between spatial pattern and ecological responses. Since `metrics_df` is a pandas data frame, we can easily compute the correlation matrix of the metrics and plot it as a heat map:

In [ ]:
sns.heatmap(ten_ssa.metrics_df.corr(), **heatmap_kwargs)

The heatmap shows that many metrics are almost perfectly correlated, either positively (e.g., edge density and landscape shape index) or negatively (e.g., contagion and Shannon diversity index).

One way to address this issue is to factorize the metrics data frame into a reduced set of components that explain the most variance in the data. To that end, the scikit-learn library [features many classes implementing different decomposition algorithms](https://scikit-learn.org/stable/modules/decomposition.html#decompositions). Let us use the [Principal Component Analysis (PCA)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) algorithm:

In [ ]:
# provide `random_state` for reproducibility
component_df, decompose_model = ten_ssa.decompose(
    decomposer=decomposition.PCA, random_state=random_seed
)
component_df.head()

The `decompose` method returns both (i) a data frame with the components as columns and the landscapes as rows and (ii) the decomposition model. While the data frame alone may be hard to interpret, it can be used in conjunction with the decomposition model to obtain very useful information, such as the explained variance of each component and the loadings of each metric on each component.

We can access the explained variance of each component by using the `explained_variance_ratio_` attribute of the decomposition model:

In [ ]:
decompose_model.explained_variance_ratio_

As we can see, the four first components respectively explain a 64.94, 17.38, 12.58 and 2.41 % of the total variance (making it a total of 97.31 %). This information can be used to decide how many components to retain in the analysis.

We can also access the loadings of each metric on each component by using the `get_loading_df` method:

In [ ]:
loading_df = ten_ssa.get_loading_df(decompose_model)
loading_df

Given the explained variance, we can focus on the first four components and visualize this information using a heatmap:

In [ ]:
# ten_ssa.plot_loading_heatmap(decompose_model)
n_components = 4
sns.heatmap(loading_df.iloc[:, :n_components], **heatmap_kwargs)

As we can see, the first component is positively correlated with edge density and landscape shape index and negatively correlated with contagion, which suggest that this component is negatively related to the aggregation of patches. The second, third and fourth components are positively correlated with the mean shape index, area-weighted mean fractal dimension and mean patch area metrics.

In short, the component analysis can help identifying the fundamental components of spatial patterns and their relationships with the landscape metrics. This can help us in choosing the most relevant metrics for a given application and avoid multicolinearity issues.

## Clustering landscapes based on spatial signatures

Another common application of the spatial signature analysis is to cluster similar landscapes based on their spatial patterns. To assist in this task, the `SpatialSignatureAnalysis` class provides a `get_cgram` method that computes a clustergram [6] based on the spatial signagures. The returned object is a [`Clustergram`](https://clustergram.readthedocs.io/en/stable/api.html#clustergram.clustergram.Clustergram) [7] instance that can be used to visualize clustergram diagrams and select the number of clusters:

In [ ]:
# provide `random_state` for reproducibility
ten_cgram = ten_ssa.get_cgram(k_range=range(2, 10), random_state=random_seed)
ten_cgram.plot()

As we can see, setting the number of clusters to 3 seems a good choice. Alternatively, we can use the `silhouette_score`, `calinski_harabasz_score` and `davies_bouldin_score` methods of the returned `cgram` to evaluate the quality of the clustering for different numbers of clusters:

In [ ]:
_ = plot_cgram_eval(ten_cgram)

Ideally, we want to pick the number of clusters that maximizes the `silhouette_score` and `calinski_harabasz_score` and minimizes the `davies_bouldin_score`, which in this case is in line with the clustergram diagram, i.e., 3 or 4 clusters.

Given the number of clusters, we can access the labels of the landscapes in each cluster by using the `labels_` attribute of the `cgram` object:

In [ ]:
n_clusters = 4
ten_cgram.labels_[n_clusters]

We can also use `scatterplot_cluster_metrics` method of `SpatialSignatureAnalysis` to obtain a scatterplot of any given pair of metrics, with the landscapes colored by their cluster labels. Based on the metrics correlations and PCA loadings, we can choose the contagion, mean shape index, area-weighted mean fractal dimension and mean area metrics to visualize the clusters:

In [ ]:
other_metrics = ["shape_index_mn", "fractal_dimension_am", "area_mn"]
fig, axes = plt.subplots(
    1,
    len(other_metrics),
    figsize=(len(other_metrics) * figwidth, figheight),
    sharey=True,
)
for other_metric, ax in zip(other_metrics, axes):
    ten_ssa.scatterplot_cluster_metrics(
        ten_cgram, n_clusters, other_metric, "contagion", ax=ax
    )

As we can see, the clusters can be largely separated by means of the contagion index only, with cluster "0" having low values, clusters "1" and "3" having mid-range values and cluster "2" having high values Additionally, clusters "2" and "3" are characterized by lower values of the area-weighted mean fractal dimension and higher values of mean patch area. *Note that the colored "x" markers correspond to the centroids of their respective clusters.*

To get a better grasp of the clustering results, we can use the `plot_cluster_landscapes` method to visualize the landscapes in each cluster:

In [ ]:
ten_fig = ten_ssa.plot_cluster_landscapes(
    ten_cgram, n_clusters, **plot_cluster_landscapes_kwargs
)

Finally, we can use the `plot_cluster_zones` method to visualize the landscape extents colored by their cluster labels:

In [ ]:
ten_ssa.plot_cluster_zones(ten_cgram, n_clusters)

*Note that we can only call `plot_cluster_zones` if the `SpatialSignatureAnalysis` instance has been initialized with a zonal analysis class (i.e., `ZonalAnalysis`, `BufferAnalysis`, `ZonalGridAnalysis` and their corresponding spatio-temporal analsysis classes).*

## Fundamental components of landscape patterns: further insights from information theory (IT)

The above sections illustrate how the `SpatialSignatureAnalysis` class operates with some example real-world applications. Let us now take this one step further and address a key question of spatial pattern analysis in landscape ecology (see the "Spatial patterns" section of the review of Hesselbarth et al. [3] for more details): *what are the fundamental components of landscape configuration?*

Following the approach of Nowosad and Stepinski [2], we will now try to use an information theory (IT)-based approach to classify landscapes, i.e., the HYU diagram, where H and U respectively refer to [the Shannon's entropy](https://pylandstats.readthedocs.io/en/latest/landscape.html#pylandstats.Landscape.entropy) and [the relative mutual information criterion](https://pylandstats.readthedocs.io/en/latest/landscape.html#pylandstats.Landscape.relative_mutual_information) (see Nowosad and Stepinski [2] for more details).

To perform this IT-based analysis, let us instantiate another `SpatialSignatureAnalysis` using these two metrics (which can only be computed at the landscape level):

In [ ]:
it_metrics = ["entropy", "relative_mutual_information"]
it_ssa = pls.SpatialSignatureAnalysis(za, landscape_metrics=it_metrics)

Note that since we are only using two landscape metrics, we can skip the factorization into components (e.g., PCA) and proceed directly to the cluster analysis.

In [ ]:
it_cgram = it_ssa.get_cgram(k_range=range(2, 10), random_state=random_seed)
it_cgram.plot()

Likewise the analysis based on ten landscape metrics, we can complement the clustergram with the other metrics:

In [ ]:
_ = plot_cgram_eval(it_cgram)

The picture here is a bit more nuanced but the metrics seemingly agree that using 2 or 7 clusters is an appropriate choice. We can again visualize the clusters in a two-dimensional scatter plot of the two metrics:

In [ ]:
n_clusters = 2
_ = it_ssa.scatterplot_cluster_metrics(
    it_cgram, n_clusters, "entropy", "relative_mutual_information"
)

It seems that the clusters can be linearly separated but in this case this requires considering the two metrics. Let us now visualize the landscapes of each cluster:

In [ ]:
it_fig = it_ssa.plot_cluster_landscapes(
    it_cgram, n_clusters, **plot_cluster_landscapes_kwargs
)

We can see that the *cluster 1* seems to group the landscapes with most uneven distribution of class abundance (hence the lowest entropy values).

To conclude, let us compare the results of the ten metrics-based and the IT-based cluster analyses. We can evaluate the consistency of the ten metrics-based and the IT-based cluster classifications by comparing their respective [silhouette scores](https://en.wikipedia.org/wiki/Silhouette_(clustering)):

In [ ]:
colors = sns.color_palette()
fig, ax = plt.subplots()
for cgram, label, color in zip([ten_cgram, it_cgram], ["Ten metrics", "IT"], colors):
    cgram.silhouette_score().plot(color=color, ax=ax, label=label)
ax.legend()
ax.set_xlabel("n. clusters")
ax.set_ylabel("silhouette score")

We can see that using ten metrics consistently results in lower silhouette scores than using the IT-based approach. Additionally, it is worth noting that using the ten metrics is very likely to result in multi-collinearity issues.

Finally, let us plot the landscape extents colored by their cluster labels on a map:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(figwidth * 2, figheight))
for ssa, cgram, title, ax in zip(
    [ten_ssa, it_ssa], [ten_cgram, it_cgram], ["Ten metrics", "IT"], axes
):
    ssa.plot_cluster_zones(cgram, n_clusters, ax=ax)
    ax.set_title(title)

## Bonus track: spatial signatures in a spatio-temporal zonal analysis

Let us conclude by combining the spatial signatures with a spatio-temporal zonal analysis:

In [ ]:
lulc_cols = ["LU85_4", "LU97_4", "LU09_4", "LU18_4"]
input_filepaths = [f"data/vaud/{lulc_col}.tif" for lulc_col in lulc_cols]
# note that these are the "survey" dates but each survey takes a total of 5 years so the
# actual date of each pixel depends on the region - see the "Swiss Land Use Statistics"
# documentation at https://shorturl.at/FMESv
dates = ["1985", "1997", "2009", "2018"]

stza = pls.SpatioTemporalZonalAnalysis(
    input_filepaths, zga.zone_gser[is_inner].copy(), dates=dates
)
stza_ssa = pls.SpatialSignatureAnalysis(stza, landscape_metrics=it_metrics)

The `SpatialSignatureAnalysis` class work seamlessly, e.g., we can see the computed metrics (note that in this case, the landscapes are indexed by both the grid zone identifier and date):

In [ ]:
stza_ssa.metrics_df

We can also decompose the computed metrics into components as shown above, however it does not make sense since we are using the IT-based approach. Let us cluster the landscapes instead:

In [ ]:
stza_cgram = stza_ssa.get_cgram(k_range=range(2, 10), random_state=random_seed)
stza_cgram.plot()
_ = plot_cgram_eval(stza_cgram)

It seems that in this case considering 6 clusters can be more appropriate.

In [ ]:
n_clusters = 6

Let us now plot the landscape clusters based on their metrics' values:

In [ ]:
_ = stza_ssa.scatterplot_cluster_metrics(
    stza_cgram, n_clusters, "entropy", "relative_mutual_information"
)

Again, the landscapes seem linearly separable when considering the two metrics.

Finally, the only difference when using a spatio-temporal zonal analysis class comes when plotting the cluster zones:

In [ ]:
_ = stza_ssa.plot_cluster_zones(stza_cgram, n_clusters)

We can now visualize how the landscape clusters based on the IT metrics change over time.

Even though this is beyond the scope of this notebook, note that we have only included metrics of spatial configuration at the landscape level. Spatial abundance (namely the proportion of landscape metric at each class level) is certainly another fundamental component of spatial pattern, if not the most fundamental one [3, 5, 8-10]. In the IT-based approach, spatial abundance is likely reflected in the entropy metric, but it may be interesting to explicitly consider spatial abundance. Additionally, abundance of specific LULC classes can be a good predictor of many downstream applications.

## References

1. Nowosad, Jakub. "Motif: an open-source R tool for pattern-based spatial analysis." Landscape Ecology 36 (2021): 29-43.
2. Nowosad, J., & Stepinski, T. F. (2019). Information theory as a consistent framework for quantification and classification of landscape patterns. Landscape Ecology, 34(9), 2091-2101.
3. Hesselbarth, M. H., Nowosad, J., de Flamingh, A., Simpkins, C. E., Jung, M., Gerber, G., & Bosch, M. (2025). Computational Methods in Landscape Ecology. Current Landscape Ecology Reports, 10(1), 1-18.
4. Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., ... & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. the Journal of machine Learning research, 12, 2825-2830.
5. Nowosad, J., & Stepinski, T. F. (2018). Global inventory of landscape patterns and latent variables of landscape spatial configuration. Ecological Indicators, 89, 159-167.
6. Schonlau, M. (2002). The clustergram: A graph for visualizing hierarchical and nonhierarchical cluster analyses. The Stata Journal, 2(4), 391-402.
7. Fleischmann, M. (2023). Clustergram: Visualization and diagnostics for cluster analysis. Journal of Open Source Software, 8(89), 5240.
8. Gustafson, E. J. (1998). Quantifying landscape spatial pattern: what is the state of the art?. Ecosystems, 1(2), 143-156.
9. Gustafson, E. J. (2019). How has the state-of-the-art for quantification of landscape pattern advanced in the twenty-first century?. Landscape Ecology, 34, 2065-2072.
10. Riitters, K. (2019). Pattern metrics for a transdisciplinary landscape ecology. Landscape Ecology, 34(9), 2057-2063.